# Send agent traces to MLflow

This notebook deploys `tracing-assistant`, an agent that sends its traces to an MLflow experiment through a collector in your workspace. See the [README](./README.md) for the same steps in a terminal and for enabling tracing on an existing agent.

kagent must be enabled on your prokube cluster, and its MLflow must accept OTLP traces. See the README's [Prerequisites](./README.md#prerequisites).

## Setup

Set up this repository's kagent helpers from [kagent_helpers.py](../../shared/kagent_helpers.py) and the MLflow credential helpers from [mlflow_credentials.py](../../../src/pk_helpers/mlflow_credentials.py). Neither needs to be installed.

In [ ]:
import sys

sys.path.insert(0, "../../shared")  # kagent helpers (shared by all kagent examples)

import kagent_helpers

%run -n ../../../src/pk_helpers/mlflow_credentials.py

In [ ]:
import os
import subprocess
import time
from pathlib import Path

import mlflow
from mlflow import MlflowClient

with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace") as f:
    namespace = f.read().strip()

## Prerequisites

The agent uses the `anthropic-haiku` ModelConfig from [`basic-agent`](../../basic-agent/). The next cell creates it and the `anthropic-api-key` Secret it reads, like the earlier kagent notebooks:

- `ensure_anthropic_secret()` skips a Secret that already exists and otherwise prompts for the key unless `ANTHROPIC_API_KEY` is set.
- `ensure_model_config()` applies [`anthropic-model.yaml`](../../basic-agent/anthropic-model.yaml) and waits until kagent accepts it.

In [ ]:
kagent_helpers.ensure_anthropic_secret()
kagent_helpers.ensure_model_config()

## Create the MLflow credentials

The collector authenticates to MLflow with the `mlflow-credentials` Secret. First, [create a Personal Access Token in MLflow](/mlflow/oidc/ui/auth) and copy it. Then run the next cell. It prompts for the MLflow URL, your username, and the token, and creates the Secret.

If the Secret exists, for example from the MLflow examples, setup is skipped. Use `rerun=True` to replace it. Automated tests skip this interactive cell.

In [ ]:
# Creates the Secret if it doesn't exist.
setup_mlflow_credentials()

# To recreate the secret with new credentials, run:
# setup_mlflow_credentials(rerun=True)

Load the same credentials into this notebook, so that it can create the experiment and read the traces:

In [ ]:
load_mlflow_credentials()

## Choose an experiment

The next cell creates the experiment `kagent tracing <your username>`, or reuses it, and reads its ID. To send the traces somewhere else, change the name. Your account needs **EDIT** permission on the experiment, which it has on experiments it creates.

In [ ]:
username = os.environ["MLFLOW_TRACKING_USERNAME"].split("@")[0]
experiment = mlflow.set_experiment(f"kagent tracing {username}")
experiment_id = experiment.experiment_id
print(f"Experiment {experiment.name!r} has ID {experiment_id}")

## How it works

[`assistant.yaml`](./assistant.yaml) is the assistant from [`basic-agent`](../../basic-agent/) with tracing enabled. It uses the Python runtime and sends its traces to a collector in the workspace:

```yaml
spec:
  declarative:
    runtime: python
    deployment:
      env:
      - name: OTEL_TRACING_ENABLED
        value: "true"
      - name: OTEL_EXPORTER_OTLP_TRACES_ENDPOINT
        value: http://otel-collector:4317
      - name: OTEL_EXPORTER_OTLP_TRACES_PROTOCOL
        value: grpc
```

[`otel-collector.yaml`](./otel-collector.yaml) deploys an [OpenTelemetry Collector](https://opentelemetry.io/docs/collector/) that forwards the traces to MLflow at `<MLFLOW_TRACKING_URI>/v1/traces`. It authenticates with the `mlflow-credentials` Secret and selects the experiment through its `MLFLOW_EXPERIMENT_ID` variable.

## Deploy the collector

`otel-collector.yaml` has a placeholder for the experiment ID. The next cell fills in the ID of your experiment and applies the manifest. If the workspace already runs an `otel-collector` that someone else manages, skip this cell and reuse that collector.

In [ ]:
manifest = Path("otel-collector.yaml").read_text().replace("<experiment-id>", experiment_id)
result = subprocess.run(["kubectl", "apply", "-f", "-"], input=manifest, capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"kubectl apply failed: {result.stderr.strip()}")
print(result.stdout, end="")

!kubectl rollout status deployment/otel-collector --timeout=2m

## Deploy the agent

In [ ]:
!kubectl apply -f assistant.yaml
!kubectl wait --for=condition=Ready agents.kagent.dev/tracing-assistant --timeout=3m

## Talk to the agent

Send a message through the internal Agent Gateway route, as in the earlier kagent notebooks. Traces contain prompts and answers, so do not send sensitive data. You can also open the [Agents page](/pkui/kagent) in the prokube UI (`https://<your-prokube-domain>/pkui/kagent`), select `tracing-assistant`, and chat there.

In [ ]:
internal_url = kagent_helpers.workspace_a2a_url(namespace, "tracing-assistant")
# A few seconds earlier, in case the clocks of this Lab and the agent's pod differ slightly.
sent_ms = time.time_ns() // 1_000_000 - 5_000

prompt = "What can you help me with?"
print(kagent_helpers.send_message(internal_url, prompt))

## Find the trace

The agent and the collector send traces in batches, so the trace arrives a few seconds after the answer. The next cell waits for it and lists its spans. Expect spans for the agent's request and for its call to the model.

In [ ]:
client = MlflowClient()
deadline = time.monotonic() + 120
while True:
    traces = client.search_traces(locations=[experiment_id], order_by=["timestamp_ms DESC"], max_results=1)
    if traces and traces[0].info.timestamp_ms >= sent_ms:
        break
    if time.monotonic() > deadline:
        raise RuntimeError("No trace arrived within 2 minutes. Check the collector logs with: kubectl logs deployment/otel-collector")
    time.sleep(5)

trace = traces[0]
print(f"Trace {trace.info.trace_id}")
for span in trace.data.spans:
    print(f"  {span.span_type or '-':<10} {span.name}")

Open the experiment's traces in MLflow to inspect the request, the answer, and the timing of each span:

In [ ]:
print(f"{os.environ['MLFLOW_TRACKING_URI'].rstrip('/')}/#/experiments/{experiment_id}/traces")

## Clean up

Uncomment the command and run the cell to delete the agent and the collector. This leaves the ModelConfig and the `anthropic-api-key` Secret, which the other kagent examples use, and the `mlflow-credentials` Secret, which the MLflow examples use. The traces stay in the MLflow experiment.

In [ ]:
# !kubectl delete -f assistant.yaml -f otel-collector.yaml